In [ ]:
import sys
sys.path.append("../")

from src.service.elements import ElementsLLMUtils

llm = ElementsLLMUtils().get_opensrc_llm()

/Users/vn573wh/work/IntraGenAI-serv/experiments/../envs/ca-bundle.crt


/Users/vn573wh/work/IntraGenAI-serv/experiments/../src/service/elements.py:39: LangChainDeprecationWarning: The class `HuggingFaceTextGenInference` was deprecated in LangChain 0.0.21 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEndpoint`.
  return HuggingFaceTextGenInference(
/Users/vn573wh/work/IntraGenAI-serv/.venv/lib/python3.11/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_id" in DeployedModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/vn573wh/work/IntraGenAI-serv/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stabl

In [1]:
import pandas as pd
from langchain_community.llms import HuggingFaceTextGenInference  ## Working 

In [4]:

from langchain.agents import initialize_agent, Tool

def get_node_rules(node_rule):

    node_rule_dict = {
        "is_small_oms_gbm_score_accept" : "varA = (_v('gbmscore_accept') < 0.0006);",

        "is_home_bot_deny" : '''
varA = (_v('max_dispense_type') == "HOME");
varB = (_v('max_pi_hash_count_can_order_1d') >= 3);
varC = !/YY/.test(_v('max_avs_code'));
varD = (_dv_is_tmx_summary_code_bot == true);
returnA && B && C && D;
''',

        "is_small_order_amount_accept" : '''
varA = (_v('order_amount') < 200);
varB = (_v('customer_days_in_file') > 30);
varC = (_v('customer_days_in_file') <= 700);
varD = (_v('can_cb_count_order_by_cust_30d') == 0);
varE = (_v('ratio_cust_count_can_order_7_30d') < 10);
varF = (_v('cust_sum_can_order_amt_30d') < 400);
varG = (_v('ratio_cust_count_can_order_14_30d') < 100);
varH = (_v('max_credential_check1') == "CRY");
varI = (_v('max_dispense_type') == "GROCERIES_HOME");
varJ = (_v('cust_sum_can_order_amt_30d') > 0);
returnA && B && C && D && E && F && G && H && I && J;
''',
        "is_tmx_risky_reason_code_deny" : '''

varA = (_dv_is_tmx_summary_code_risky == true);
varB = _.includes(tmx_risky_code_credential_check, _v('max_credential_check1'));
varC = (_v('order_amount') >= 450);
varD = (_v('can_bankauth_pihash_riskyerror_count_72h') >= 1);
returnA && B && C && D;''',
        "is_high_gbm_score_bot_deny" : '''

varA = (_dv_is_tmx_summary_code_bot == true);
varB = (_v('gbmscore_oms_accept') >= 0.9);
varC = (_v('can_bankauth_cust_error_count_30d') >= 5);
varD = (_v('order_amount') >= 100);
returnA && B && C && D;''',

        "is_high_bankauth_error_24h_deny" : '''
varA = (_v('can_bankauth_cust_error_count_24h') >= 3);
varB = (_v('max_credential_check1') != "CRY");
varC = !_.includes(high_bankauth_error_24h_avs_code, _v('max_avs_code'));
varD = (_v('gbmscore_oms_accept') >= 0.8);
returnA && B && C && D;''',

        "is_electronic_bulkybuy_deny" : '''
varA = (_v('max_dispense_type') == "ELECTRONIC");
varB = !/YY/.test(_v('max_avs_code'));
varC = (_v('cust_count_can_order_1d') >= 2);
varD = (_v('max_pm_id') != "FDCGC");
varE = (_v('max_pm_id') != "KLARNA");
returnA && B && C && D && E;''',

        "large_monthly_order_deny" : '''
varA = (_v('gbmscore_oms_accept') >= 0.983);
varB = (_v('cust_sum_can_order_amt_30d') >= 5300);
varC = (_v('cust_count_can_order_1d') >= 4);
varD = (_v('can_bankauth_cust_error_count_24h') >= 1);
returnA && B && C && D;'''
    }
    return node_rule_dict.get(node_rule, "No rules available for the given node!.")

# Function to get variable descriptions
def get_variable_dict(variable: str): 
    variable = variable.strip("_v(").strip(")")
    variable_dict = {
    'gbmscore_oms_accept': "the risky score based on our machine learning model, predicting the possibility that the order falls into the safe bucket.",
    'order_amount': "total order amount in CAD.",
    'customer_days_in_file': "how long the customer has been Walmart's customer.",
    'can_cb_count_order_by_cust_30d': "How many orders placed by this cust_id are chargeback orders in the past 30 days.",
    'ratio_cust_count_can_order_7_30d': "Ratio compare 1)how many orders placed by this cust_id in the past 7 days; 2)how many orders placed by this cust_id in the past 30 days.",
    'cust_sum_can_order_amt_30d': "Total order amount in CAD placed by this cust_id in past 30 days.",
    'ratio_cust_count_can_order_14_30d': "Ratio compare 1)how many orders placed by this cust_id in the past 14 days; 2)how many orders placed by this cust_id in the past 30 days.",
    'max_credential_check1': "payment credential check for the payment of line items with the maximum amount.",
    'max_dispense_type': "delivery option for the line itmes with the maximum amount.",
    'max_avs_code': "Address Verification Service code for the payment of line items with the maximum amount.",
    'cust_count_can_order_1d': "How many orders placed by this cust_id in the past 1 day.",
    'max_pm_id': "payment method ID for the payment of line items with the maximum amount.",
    'max_pi_hash_count_can_order_1d': "pihash is the unique and tokenised identifier generated for any tender card(CC/GC/Paypal,etc) used for paying for order on Walmart Ecomm site. This feature refers to the number of orders placed by this pihash in the past 1 day.",
    '_dv_is_tmx_summary_code_bot': "Suggested by ThreatMetrix data, this order is likely to be placed by bot.",
    '_dv_is_tmx_summary_code_risky': "Suggested by ThreatMetrix data, this order is likely to be a risky one.",
    'tmx_risky_code_credential_check': "payment credential check for the payment of line items with the maximum amount falls into riksy one.",
    'can_bankauth_pihash_riskyerror_count_72h': "pihash is the unique and tokenised identifier generated for any tender card(CC/GC/Paypal,etc) used for paying for order on Walmart Ecomm site. This feature refers to the number of orders placed by this pihash in the past 72 hours, but got denied by bank authorization. The denial reason is related to risky denial code.",
    'can_bankauth_cust_error_count_24h': 'The number of orders placed by this cust_id in the past 24 hours, but got denied by bank authoration.',
    'high_bankauth_error_24h_avs_code': "The Address Verification Service code for the payment of line items with the maximum amount falls into risky one.",
    'can_bankauth_cust_error_count_30d': "The number of orders placed by the cust but got denied by bank authorization in the past 30 days."
    }
    return variable_dict.get(variable, "No description available for the given variable.")

# Function to explain the code logic
def explain_code(code: str):
    return f"""
   Based on the rules, identify the variables in the code, and get the variable description using variable_dict_tool. 
    
   Example:
    	
var A = (_dv_is_tmx_summary_code_bot == true);
var B = (_v('gbmscore_oms_accept') >= 0.9);
var C = (_v('can_bankauth_cust_error_count_30d') >= 5);
var D = (_v('order_amount') >= 100);
return A && B && C && D;

    Explanation:

    Based on our model, the customer was identified as risky to initiate chargebacks in the future. The customer's order was denied because all of the following conditions are met.
    
    - If suggested by ThreatMetrix data, this order is likely to be one placed by bot.
    - If based on the score from our machine learning model, the order is predicted to be risky.
    - If the number of orders placed by the cust but got denied by bank authorization in the past 30 days is higher or equal to 5.
    - The order amount is greater than or equal to 100 CAD.
    

    Using the description to explain why this customer's order is denied {code}
    """

node_rules_dict_tool = Tool(
    name="NodeRule",
    func=get_node_rules,
    description="Retrieve the rules for a given node."
)

# Define tools for the agent
variable_dict_tool = Tool(
    name="VariableDict",
    func=get_variable_dict,
    description="Retrieve the description of a given variable."
)

explain_code_tool = Tool(
    name="ExplainCode",
    func=explain_code,
    description="Explain why the customer was denied by the rules."
)

# Define the LLM
# llm = ChatOpenAI(temperature=0, model="gpt-4")

# Initialize the agent with tools
tools = [variable_dict_tool, explain_code_tool, node_rules_dict_tool]
agent = initialize_agent(tools, llm, agent="zero-shot-react-description", verbose=False)



# Run the agent with the input
response = agent.run(f"Explain why the customer was denied for the is_high_gbm_score_bot_deny rule")
print(response)


The customer's order was denied because the customer's order was identified as risky based on the ThreatMetrix data, GBM score, bank authorization error count, and the order amount.


In [33]:
input_code = """
var A = (_v('max_dispense_type') == "HOME");
var B = (_v('max_pi_hash_count_can_order_1d') >= 3);
var C = !/YY/.test(_v('max_avs_code'));
var D = (_dv_is_tmx_summary_code_bot == true);
return A && B && C && D;
"""

# Run the agent with the input
response = agent.run(f"Explain why the customer was denied by the rules: {input_code}")
print(response)

The customer's order was denied because all of the following conditions are met: 
- The delivery option for the line items with the maximum amount is "HOME". 
- The number of orders placed by this pihash in the past 1 day is 3 or more. 
- The Address Verification Service code for the payment of line items with the maximum amount does not contain "YY". 
- The ThreatMetrix data suggests this order is likely to be one placed by bot.


In [36]:
input_code = """
var A = (_v('order_amount') < 200);
var B = (_v('customer_days_in_file') > 30);
var C = (_v('customer_days_in_file') <= 700);
var D = (_v('can_cb_count_order_by_cust_30d') == 0);
var E = (_v('ratio_cust_count_can_order_7_30d') < 10);
var F = (_v('cust_sum_can_order_amt_30d') < 400);
var G = (_v('ratio_cust_count_can_order_14_30d') < 100);
var H = (_v('max_credential_check1') == "CRY");
var I = (_v('max_dispense_type') == "GROCERIES_HOME");
var J = (_v('cust_sum_can_order_amt_30d') > 0);
return A && B && C && D && E && F && G && H && I && J;
"""

# Run the agent with the input
response = agent.run(f"Explain why the customer was denied by the rules: {input_code}")
print(response)

The customer's order was denied because all of the following conditions are met. 
- The total order amount is less than 200 CAD.
- The customer has been with Walmart for more than 30 days.
- The customer has been with Walmart for 700 days or less.
- The customer has no chargebacks in the past 30 days.
- The customer's ordering frequency in the past 7 days compared to the past 30 days is less than 10.
- The total order amount in the past 30 days is less than 400 CAD.
- The customer's ordering frequency in the past 14 days compared to the past 30 days is less than 100.
- The payment credential check is "CRY".
- The delivery option is "GROCERIES_HOME".
- The total order amount in the past 30 days is more than 0 CAD.


In [37]:

input_code = """
var A = (_dv_is_tmx_summary_code_risky == true);
var B = _.includes(tmx_risky_code_credential_check, _v('max_credential_check1'));
var C = (_v('order_amount') >= 450);
var D = (_v('can_bankauth_pihash_riskyerror_count_72h') >= 1);
return A && B && C && D;
"""

# Run the agent with the input
response = agent.run(f"Explain why the customer was denied by the rules: {input_code}")
print(response)

Based on our model, the customer was identified as risky. The customer's order was denied because all of the following conditions are met.

- If suggested by ThreatMetrix data, this order is likely to be risky.
- If the payment credential check for the payment of line items with the maximum amount falls into risky one.
- If the total order amount in CAD is greater than or equal to 450.
- If the number of orders placed by this pihash in the past 72 hours, but got denied by bank authorization with risky denial code, is higher or equal to 1.
